INSTALACIÓN DE LIBRERIAS

In [1]:
%pip install pandas pyarrow plotly kaleido jinja2 python-docx weasyprint

Note: you may need to restart the kernel to use updated packages.


IMPORTACIÓN DE ARCHIVOS Y CONFIGURACIÓN DE RUTAS

In [2]:
import os, re, json, base64, hashlib, shutil, subprocess, math, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import plotly.graph_objects as go
from jinja2 import Environment, FileSystemLoader

warnings.filterwarnings("ignore")

# Parámetros editables
RUTA_MANUAL = None   
VERSION = os.environ.get("REPORTE_VERSION", "1.0")
EQUIPO = "Equipo 5 — Cybersecurity Data Analytics"
FUENTE_TXT = "dataset_enriquecido_final.parquet (CICIDS2017, agregado y anonimizado)"

# Carpeta del proyecto de reportería
if os.environ.get("REPORTE_DIR"):
    BASE = Path(os.environ["REPORTE_DIR"])
elif (Path.cwd() / "datos" / "agregados").exists():  
    BASE = Path.cwd()
else:
    BASE = Path("..") / "Reporteria"                      

CARPETAS = ["datos/agregados", "evidencias/reporte", "exportaciones", "templates", "documentos", ".github/workflows"]
for c in CARPETAS:
    (BASE / c).mkdir(parents=True, exist_ok=True)

AGG, IMG, OUT, TPL = BASE / "datos/agregados", BASE / "evidencias/reporte", BASE / "exportaciones", BASE / "templates"
print("Carpeta del proyecto:", BASE.resolve())

Carpeta del proyecto: /home/vctor/Proyecto_GDIABD_Cybersecurity_Data_Analytics/Reporteria


LOCALIZAR EL PARQUET

In [3]:
def localizar_parquet():
    if RUTA_MANUAL:
        return Path(RUTA_MANUAL)
    if os.environ.get("PARQUET_ENRIQUECIDO"):
        return Path(os.environ["PARQUET_ENRIQUECIDO"])
    candidatos = []
    try:
        from Configuracion import ANALITICOS_PARQUET            
        candidatos.append(Path(ANALITICOS_PARQUET) / "dataset_enriquecido_final.parquet")
    except Exception:
        pass
    candidatos += [Path("..") / "Datos" / "Analiticos" / "Parquet" / "dataset_enriquecido_final.parquet",
                        Path("dataset_enriquecido_final.parquet")]
    for c in candidatos:
        if c.exists():
            return c
    for c in Path("..").rglob("dataset_enriquecido_final.parquet"):
        return c
    return None

RUTA_PARQUET = localizar_parquet()
HAY_AGREGADOS = (AGG / "resumen_general.csv").exists()
if RUTA_PARQUET is not None:
    MODO = "parquet"
elif HAY_AGREGADOS:
    MODO = "agregados"
else:
    raise FileNotFoundError("No se encontró el parquet enriquecido ni agregados. Define RUTA_MANUAL en la celda 2.")
print("MODO:", MODO, "|", RUTA_PARQUET if MODO == "parquet" else AGG)

MODO: parquet | ../Datos/Analiticos/Parquet/dataset_enriquecido_final.parquet


CONSTRUCCIÓN DE AGREGADOS, SOLO LAS COLUMNAS QUE SEAN NECESARIAS

In [4]:
FAMILIAS = {
    "BENIGN": "Benigno",
    "DoS Hulk": "DoS/DDoS", "DoS GoldenEye": "DoS/DDoS", "DoS Slowhttptest": "DoS/DDoS",
    "DoS slowloris": "DoS/DDoS", "DDoS": "DoS/DDoS",
    "PortScan": "Reconocimiento",
    "FTP-Patator": "Fuerza bruta", "SSH-Patator": "Fuerza bruta",
    "Web Attack - Brute Force": "Web Attack", "Web Attack - XSS": "Web Attack", "Web Attack - Sql Injection": "Web Attack",
    "Bot": "Botnet", "Infiltration": "Infiltración", "Heartbleed": "Vulnerabilidad crítica",
}
COLUMNAS = ["label", "timestamp", "destination_port", "protocol", "flow_duration",
            "volumen_paquetes", "categoria_duracion", "es_bidireccional", "ratio_fwd_bwd_bytes"]

#Unifica el guion de 'Web Attack' (corrige \x96 / 'ï¿½' / '–' por si quedó el problema de codificación)
def limpiar_label(s):
    return s.astype(str).str.strip().str.replace(r"\s*(?:\x96|ï¿½|–)\s*", " - ", regex=True)

def construir_agregados(ruta):
    d = ds.dataset(str(ruta), format="parquet", partitioning="hive")
    df = d.to_table(columns=COLUMNAS).to_pandas()
    df["label"] = limpiar_label(df["label"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["es_ataque"] = df["label"] != "BENIGN"
    df["familia"] = df["label"].map(FAMILIAS).fillna("Otro")
    df["fecha"] = df["timestamp"].dt.date
    df["hora"] = df["timestamp"].dt.hour
    df["protocolo"] = df["protocol"].map({6: "TCP", 17: "UDP"}).fillna("Otro")
    df["duracion_seg"] = df["flow_duration"] / 1e6            # CICFlowMeter reporta microsegundos
    if (df["familia"] == "Otro").any():
        print("AVISO: etiquetas sin familia:", df.loc[df["familia"] == "Otro", "label"].unique())

    tab = lambda cols: (df.groupby(cols + ["es_ataque"]).size().unstack(fill_value=0)
                        .reindex(columns=[False, True], fill_value=0)
                        .rename(columns={False: "benigno", True: "ataque"}).reset_index())
    a, b = df[df.es_ataque], df[~df.es_ataque]
    out = {
        "resumen_general": pd.DataFrame([{
            "fuente": Path(ruta).name, "total_flujos": len(df), "ataques": int(df.es_ataque.sum()),
            "fecha_min": df.timestamp.min(), "fecha_max": df.timestamp.max(),
            "dur_med_benigno_s": b.duracion_seg.median(), "dur_med_ataque_s": a.duracion_seg.median(),
            "ratio_med_benigno": b.ratio_fwd_bwd_bytes.median(), "ratio_med_ataque": a.ratio_fwd_bwd_bytes.median(),
            "pct_bidir_benigno": b.es_bidireccional.mean() * 100, "pct_bidir_ataque": a.es_bidireccional.mean() * 100}]),
        "por_etiqueta": (df.groupby(["label", "familia"]).agg(flujos=("label", "size"), duracion_mediana_s=("duracion_seg", "median"))
                        .reset_index().sort_values("flujos", ascending=False)),
        "por_dia": tab(["fecha"]),
        "por_hora": tab(["hora"]),
        "puertos_ataque": (a.groupby("destination_port").size().sort_values(ascending=False).head(15).reset_index(name="ataques")),
        "por_volumen": tab(["volumen_paquetes"]),
        "por_duracion": tab(["categoria_duracion"]),
        "por_protocolo": tab(["protocolo"]),
    }
    for nombre, t in out.items():
        t.to_csv(AGG / f"{nombre}.csv", index=False)
    return out

def cargar_agregados():
    nombres = ["resumen_general", "por_etiqueta", "por_dia", "por_hora", "puertos_ataque", "por_volumen", "por_duracion", "por_protocolo"]
    return {n: pd.read_csv(AGG / f"{n}.csv") for n in nombres}

if MODO == "parquet":
    d = construir_agregados(RUTA_PARQUET)
    print("Agregados guardados en", AGG)
else:
    d = cargar_agregados()
    print("Agregados cargados desde", AGG)

# Validaciones de calidad
r = d["resumen_general"].iloc[0]
assert int(d["por_etiqueta"]["flujos"].sum()) == int(r["total_flujos"]), "Suma por etiqueta != total"
assert int(d["por_dia"][["benigno", "ataque"]].sum().sum()) == int(r["total_flujos"]), "Suma por día != total"
assert int(d["por_dia"]["ataque"].sum()) == int(r["ataques"]), "Ataques por día != ataques"
assert int(d["por_protocolo"][["benigno", "ataque"]].sum().sum()) == int(r["total_flujos"]), "Suma por protocolo != total"
print(f"Validaciones OK · {int(r['total_flujos']):,} flujos · {int(r['ataques']):,} ataques")

Agregados guardados en ../Reporteria/datos/agregados
Validaciones OK · 2,827,562 flujos · 556,555 ataques


KPIs

In [5]:
def calcular_kpis(d):
    r = d["resumen_general"].iloc[0]
    total, atq = int(r["total_flujos"]), int(r["ataques"])
    etq = d["por_etiqueta"]; solo = etq[etq["label"] != "BENIGN"].sort_values("flujos", ascending=False)
    fam = solo.groupby("familia")["flujos"].sum().sort_values(ascending=False).reset_index()
    fam["pct"] = fam["flujos"] / atq * 100
    top = solo.iloc[0]
    h = d["por_hora"].copy(); hp = h.sort_values("ataque", ascending=False).iloc[0]
    dia = d["por_dia"].copy(); dp = dia.sort_values("ataque", ascending=False).iloc[0]
    vol = d["por_volumen"].set_index("volumen_paquetes")
    pu = d["puertos_ataque"]; pr = d["por_protocolo"].set_index("protocolo")
    k = dict(
        total_flujos=total, ataques=atq, pct_malicioso=atq / total * 100, tipos_ataque=int(len(solo)),
        top_label=top["label"], top_label_flujos=int(top["flujos"]), top_label_pct=top["flujos"] / atq * 100,
        familia_top=fam.iloc[0]["familia"], familia_top_pct=float(fam.iloc[0]["pct"]),
        hora_pico=int(hp["hora"]), hora_pico_ataques=int(hp["ataque"]), hora_pico_pct=hp["ataque"] / atq * 100,
        dia_pico=str(dp["fecha"]), dia_pico_ataques=int(dp["ataque"]),
        dias_con_ataque=int((dia["ataque"] > 0).sum()), dias_total=int(len(dia)),
        pct_atq_volumen_bajo=vol.loc["Bajo", "ataque"] / atq * 100,
        pct_alto_benigno=vol.loc["Alto", "benigno"] / vol.loc["Alto"].sum() * 100,
        puerto_top=int(pu.iloc[0]["destination_port"]), puerto_top_pct=pu.iloc[0]["ataques"] / atq * 100,
        pct_atq_tcp=pr.loc["TCP", "ataque"] / atq * 100,
        dur_med_atq=float(r["dur_med_ataque_s"]), dur_med_ben=float(r["dur_med_benigno_s"]),
        ratio_med_atq=float(r["ratio_med_ataque"]), ratio_med_ben=float(r["ratio_med_benigno"]),
        fecha_min=str(r["fecha_min"])[:10], fecha_max=str(r["fecha_max"])[:10],
    )
    k["factor_duracion"] = k["dur_med_atq"] / k["dur_med_ben"] if k["dur_med_ben"] else float("nan")
    k["periodo"] = f"{datetime.fromisoformat(k['fecha_min']):%d/%m/%Y} – {datetime.fromisoformat(k['fecha_max']):%d/%m/%Y}"
    return k, fam, solo

k, fam, solo_atq = calcular_kpis(d)
fmt = lambda x: f"{x:,.0f}"

print("KPIs DEL PERIODO", k["periodo"])
print("=" * 60)
for n, v in [("Flujos analizados", fmt(k["total_flujos"])), ("Flujos de ataque", fmt(k["ataques"])),
            ("% tráfico malicioso", f"{k['pct_malicioso']:.2f}%"), ("Tipos de ataque", k["tipos_ataque"]),
            ("Ataque más frecuente", f"{k['top_label']} ({k['top_label_pct']:.1f}% de los ataques)"),
            ("Familia dominante", f"{k['familia_top']} ({k['familia_top_pct']:.1f}%)"),
            ("Hora pico", f"{k['hora_pico']}:00 ({k['hora_pico_pct']:.1f}% de los ataques)"),
            ("Ataques de bajo volumen", f"{k['pct_atq_volumen_bajo']:.1f}%"),
            ("Ataques sobre TCP", f"{k['pct_atq_tcp']:.3f}%"),
            ("Duración mediana ataque vs benigno", f"{k['dur_med_atq']:.2f}s vs {k['dur_med_ben']:.3f}s (x{k['factor_duracion']:.0f})")]:
    print(f"{n:<38}{v}")

KPIs DEL PERIODO 03/07/2017 – 07/07/2017
Flujos analizados                     2,827,562
Flujos de ataque                      556,555
% tráfico malicioso                   19.68%
Tipos de ataque                       14
Ataque más frecuente                  DoS Hulk (41.3% de los ataques)
Familia dominante                     DoS/DDoS (68.2%)
Hora pico                             16:00 (41.9% de los ataques)
Ataques de bajo volumen               70.3%
Ataques sobre TCP                     99.999%
Duración mediana ataque vs benigno    0.35s vs 0.031s (x11)


GRÁFICOS

In [6]:
PALETA = dict(azul="#1f4e79", rojo="#c0392b", gris="#7f8c8d", naranja="#e67e22")
RUTAS = {}

def guardar(fig, nombre, titulo):
    fig.update_layout(title=dict(text=titulo, x=0.01), template="plotly_white", width=1000, height=520,
                    font=dict(size=14), margin=dict(l=70, r=30, t=70, b=70))
    ruta = IMG / f"{nombre}.png"
    try:
        fig.write_image(str(ruta), scale=2)
    except Exception as e:
        try:
            import kaleido; kaleido.get_chrome_sync(); fig.write_image(str(ruta), scale=2)
        except Exception:
            raise RuntimeError("No se pudo exportar PNG. Ejecuta `plotly_get_chrome -y` en una terminal y reintenta. Detalle: " + str(e))
    RUTAS[nombre] = ruta

# G1: tipos de ataque
s = solo_atq.sort_values("flujos")
fig = go.Figure(go.Bar(x=s["flujos"], y=s["label"], orientation="h", marker_color=PALETA["rojo"],
                    text=[f"{v:,}" for v in s["flujos"]], textposition="outside"))
fig.update_xaxes(title="Flujos de red (escala log)", type="log", range=[0, math.log10(s["flujos"].max()) + 0.5])
fig.update_yaxes(title="Tipo de ataque")
guardar(fig, "g1_tipos_ataque", "Flujos maliciosos por tipo de ataque")

# G2: por día
dia = d["por_dia"]
fig = go.Figure(go.Bar(x=dia["fecha"].astype(str), y=dia["benigno"], name="Benigno", marker_color=PALETA["gris"]))
fig.add_bar(x=dia["fecha"].astype(str), y=dia["ataque"], name="Ataque", marker_color=PALETA["rojo"])
fig.update_layout(barmode="stack", legend=dict(orientation="h", y=1.08, x=0.6))
fig.update_xaxes(title="Día de captura (julio 2017)", type="category"); fig.update_yaxes(title="Flujos")
guardar(fig, "g2_ataques_por_dia", "Tráfico benigno vs. ataque por día")

# G3: por hora
h = d["por_hora"]
fig = go.Figure(go.Bar(x=h["hora"], y=h["ataque"],
                    marker_color=[PALETA["rojo"] if x == k["hora_pico"] else PALETA["azul"] for x in h["hora"]]))
fig.update_xaxes(title="Hora del día (captura)", dtick=1); fig.update_yaxes(title="Flujos de ataque")
guardar(fig, "g3_ataques_por_hora", "Ataques por hora del día (hora pico resaltada)")

# G4: familias
fig = go.Figure(go.Pie(labels=fam["familia"], values=fam["flujos"], hole=0.45,
                    marker_colors=["#c0392b", "#e67e22", "#f1c40f", "#2980b9", "#8e44ad", "#16a085", "#34495e"][:len(fam)]))
guardar(fig, "g4_familias", "Composición de los ataques por familia")

# G5: riesgo por volumen
v = d["por_volumen"].set_index("volumen_paquetes").loc[["Bajo", "Medio", "Alto"]].reset_index()
v["pct_atq"] = v["ataque"] / (v["ataque"] + v["benigno"]) * 100
fig = go.Figure(go.Bar(x=v["volumen_paquetes"], y=v["pct_atq"], marker_color=PALETA["naranja"],
                    text=[f"{x:.1f}%" for x in v["pct_atq"]], textposition="outside"))
fig.update_xaxes(title="Volumen de paquetes del flujo"); fig.update_yaxes(title="% de flujos que son ataque")
guardar(fig, "g5_volumen_riesgo", "Riesgo (% malicioso) según volumen de paquetes")

# G6: protocolo
pr = d["por_protocolo"]
fig = go.Figure(go.Bar(x=pr["protocolo"], y=pr["benigno"], name="Benigno", marker_color=PALETA["gris"]))
fig.add_bar(x=pr["protocolo"], y=pr["ataque"], name="Ataque", marker_color=PALETA["rojo"])
fig.update_layout(barmode="group", legend=dict(orientation="h", y=1.08, x=0.6))
fig.update_xaxes(title="Protocolo de transporte"); fig.update_yaxes(title="Flujos (escala log)", type="log")
guardar(fig, "g6_protocolo", "Flujos por protocolo: benigno vs. ataque")

# G7: duración mediana por tipo de ataque (+ referencia benigna)
dm = solo_atq.sort_values("duracion_mediana_s")
fig = go.Figure(go.Bar(x=dm["duracion_mediana_s"], y=dm["label"], orientation="h", marker_color=PALETA["azul"],
                    text=[(f"{x*1e6:,.0f} µs" if x < 0.001 else f"{x:,.2f} s") for x in dm["duracion_mediana_s"]], textposition="outside"))
fig.add_vline(x=k["dur_med_ben"], line_dash="dash", line_color=PALETA["gris"],
            annotation_text=f"mediana benigna {k['dur_med_ben']:.3f}s", annotation_position="bottom right")
fig.update_xaxes(title="Duración mediana del flujo (segundos, escala log)", type="log",
                range=[math.log10(max(dm["duracion_mediana_s"].min(), 1e-6)) - 0.5, math.log10(dm["duracion_mediana_s"].max()) + 0.7])
fig.update_yaxes(title="Tipo de ataque")
guardar(fig, "g7_duracion_por_ataque", "Duración mediana de los flujos por tipo de ataque")

print("Gráficos generados:", *[p.name for p in RUTAS.values()], sep="\n  ")

Gráficos generados:
  g1_tipos_ataque.png
  g2_ataques_por_dia.png
  g3_ataques_por_hora.png
  g4_familias.png
  g5_volumen_riesgo.png
  g6_protocolo.png
  g7_duracion_por_ataque.png


HALLAZGOS, RECOMENDACIONES Y LIMITACIONES

In [7]:
def construir_hallazgos(k):
    return [
        dict(titulo=f"{k['top_label']} es el ataque más frecuente",
            evidencia=f"{fmt(k['top_label_flujos'])} flujos, {k['top_label_pct']:.1f}% de todos los ataques; la familia {k['familia_top']} concentra {k['familia_top_pct']:.1f}%.",
            interpretacion="La amenaza dominante es la saturación de servicios web (denegación de servicio), no la intrusión silenciosa.",
            impacto="Un solo patrón explica la mayoría de la carga maliciosa: mitigarlo reduce el riesgo de forma desproporcionada.",
            decision="Priorizar rate limiting y alertas de DoS sobre servicios HTTP.",
            limitacion="CICIDS2017 es tráfico de laboratorio: las proporciones no son extrapolables a una red productiva."),
        
        dict(titulo=f"Los ataques se concentran en la hora {k['hora_pico']}:00",
            evidencia=f"{fmt(k['hora_pico_ataques'])} flujos de ataque en esa hora ({k['hora_pico_pct']:.1f}% del total); solo {k['dias_con_ataque']} de {k['dias_total']} días presentan ataques y el día pico fue {k['dia_pico']} ({fmt(k['dia_pico_ataques'])} ataques).",
            interpretacion="La actividad maliciosa llega en ventanas concentradas, no distribuida uniformemente.",
            impacto="Una vigilancia uniforme desperdicia capacidad analítica fuera de las ventanas críticas.",
            decision="Reforzar monitoreo y guardia en las franjas de mayor incidencia.",
            limitacion="Las horas reflejan el calendario del experimento (un escenario por franja), no un patrón humano real."),
        
        dict(titulo="Los ataques son mayormente flujos de bajo volumen de paquetes",
            evidencia=f"{k['pct_atq_volumen_bajo']:.1f}% de los ataques tienen volumen 'Bajo'; en cambio {k['pct_alto_benigno']:.2f}% de los flujos de volumen 'Alto' son benignos.",
            interpretacion="El volumen alto por sí solo no indica ataque; los ataques son flujos pequeños y repetitivos.",
            impacto="Reglas basadas solo en umbrales altos de volumen no detectarían la mayoría de los ataques.",
            decision="Combinar volumen con frecuencia, puerto destino y duración en las reglas de detección.",
            limitacion="El volumen está categorizado (Bajo/Medio/Alto): se pierde detalle fino."),
        
        dict(titulo="Los flujos de ataque duran mucho más que los benignos",
            evidencia=f"Duración mediana: {k['dur_med_atq']:.2f} s en ataques vs {k['dur_med_ben']:.3f} s en benignos (≈{k['factor_duracion']:.0f} veces más). La excepción es PortScan, con flujos de microsegundos.",
            interpretacion="Los ataques de agotamiento (DoS lentos) mantienen conexiones abiertas; el escaneo, en cambio, es ráfaga muy corta.",
            impacto="La duración separa dos comportamientos de amenaza distintos y sirve como variable de detección.",
            decision="Alertar por conexiones inusualmente largas y por ráfagas ultracortas masivas hacia muchos puertos.",
            limitacion="La mediana oculta la dispersión interna de cada tipo de ataque."),
        
        dict(titulo=f"Casi todo el ataque viaja por TCP y apunta al puerto {k['puerto_top']}",
            evidencia=f"{k['pct_atq_tcp']:.3f}% de los ataques usan TCP; el puerto {k['puerto_top']} (HTTP) recibe {k['puerto_top_pct']:.1f}% de ellos, seguido de FTP (21) y SSH (22).",
            interpretacion="La superficie de ataque expuesta es, sobre todo, el servicio web sobre TCP.",
            impacto="Proteger ese servicio cubre la mayor parte del riesgo observado.",
            decision="Hardening y monitoreo específico del servicio web; restringir FTP/SSH si no son necesarios.",
            limitacion="El puerto destino no identifica al atacante; las IP no se publican por gobernanza (Guía 4)."),
    ]

RECOMENDACIONES = [
    ("Implementar rate limiting y detección de DoS en servicios HTTP", "Alta", "Equipo de infraestructura / seguridad", "30 días", "% de flujos DoS/DDoS bloqueados"),
    ("Reforzar monitoreo en las franjas horarias de mayor incidencia", "Alta", "Analista SOC", "15 días", "Ataques detectados en hora pico / total"),
    ("Reglas de detección multivariable (volumen + frecuencia + puerto + duración)", "Media", "Ingeniería de datos / ML", "60 días", "Tasa de detección sin aumentar falsos positivos"),
    ("Restringir FTP/SSH y bloquear por intentos fallidos (fuerza bruta)", "Media", "Administración de sistemas", "45 días", "Intentos de fuerza bruta por día"),
    ("Mantener este informe automatizado y versionado en GitHub", "Baja", "Equipo 5", "Continuo", "Informes generados sin intervención manual"),
]

LIMITACIONES = [
    "Dataset de laboratorio (CICIDS2017, 3–7 jul 2017): no representa una red productiva actual.",
    "Desbalance de clases: ~80% benigno; Heartbleed (11), Infiltration (36) y SQL Injection (21) están muy poco representados.",
    "Limpieza previa: se eliminaron 288,602 flujos sin etiqueta, 203 duplicados exactos y 2,978 flujos con valores inválidos (Infinity/NaN, rangos); quedan 2,827,562.",
    "Las direcciones IP, flow_id y puertos de origen no se incluyen (minimización y seudonimización, Guía 4).",
    "No se incluye análisis geoespacial (Guía 6): es una red de laboratorio sin coordenadas reales; el equivalente por segmento de red exigiría exponer IP.",
]

hallazgos = construir_hallazgos(k)
for i, h in enumerate(hallazgos, 1):
    print(f"{i}. {h['titulo']}\n   {h['evidencia']}")

1. DoS Hulk es el ataque más frecuente
   230,123 flujos, 41.3% de todos los ataques; la familia DoS/DDoS concentra 68.2%.
2. Los ataques se concentran en la hora 16:00
   233,292 flujos de ataque en esa hora (41.9% del total); solo 4 de 5 días presentan ataques y el día pico fue 2017-07-07 (288,785 ataques).
3. Los ataques son mayormente flujos de bajo volumen de paquetes
   70.3% de los ataques tienen volumen 'Bajo'; en cambio 99.57% de los flujos de volumen 'Alto' son benignos.
4. Los flujos de ataque duran mucho más que los benignos
   Duración mediana: 0.35 s en ataques vs 0.031 s en benignos (≈11 veces más). La excepción es PortScan, con flujos de microsegundos.
5. Casi todo el ataque viaja por TCP y apunta al puerto 80
   99.999% de los ataques usan TCP; el puerto 80 (HTTP) recibe 68.7% de ellos, seguido de FTP (21) y SSH (22).


PLANTILLA HTML

In [8]:
PLANTILLA = """<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8">
<title>Informe ejecutivo de seguridad de red</title>
<style>
@page { size: A4; margin: 18mm 15mm; @bottom-center { content: "Equipo 5 · Informe v{{ version }} · Página " counter(page); font-size: 9px; color: #888; } }
body { font-family: Arial, Helvetica, sans-serif; color: #2b2b2b; line-height: 1.5; font-size: 12px; }
h2 { color: #1f4e79; margin-top: 24px; border-bottom: 1px solid #ddd; padding-bottom: 4px; page-break-after: avoid; }
h3 { margin: 0 0 6px; color: #c0392b; }
.portada { text-align: center; background: #eef4fa; padding: 22px; border-radius: 6px; margin-bottom: 18px; }
.portada h1 { color: #1f4e79; font-size: 26px; margin: 0 0 6px; }
.kpis { display: flex; flex-wrap: wrap; gap: 10px; margin: 12px 0; }
.kpi { flex: 1 1 30%; background: #eef4fa; border-left: 4px solid #1f4e79; padding: 10px; border-radius: 3px; }
.kpi .v { font-size: 20px; font-weight: bold; color: #1f4e79; } .kpi .n { font-size: 10px; color: #666; text-transform: uppercase; }
.hallazgo { background: #fafafa; border-left: 4px solid #e67e22; padding: 10px 12px; margin: 12px 0; page-break-inside: avoid; }
.hallazgo p { margin: 3px 0; }
.limit { background: #fff6dd; border-left: 4px solid #f1c40f; padding: 8px 12px; margin: 6px 0; }
table { border-collapse: collapse; width: 100%; margin: 10px 0; font-size: 11px; }
th, td { border: 1px solid #ddd; padding: 6px; text-align: left; } th { background: #1f4e79; color: #fff; }
tr:nth-child(even) { background: #f6f6f6; }
img { max-width: 100%; display: block; margin: 8px auto; } figure { margin: 10px 0; page-break-inside: avoid; }
figcaption { font-size: 10px; color: #777; text-align: center; } .meta { font-size: 10px; color: #777; }
</style></head><body>
<div class="portada">
<h1>Informe ejecutivo de seguridad de red</h1>
<div>Análisis de eventos de seguridad, accesos y alertas — CICIDS2017</div>
<p><b>Periodo:</b> {{ periodo }} · <b>Generado:</b> {{ fecha_generacion }} · <b>Versión:</b> {{ version }}<br><b>{{ equipo }}</b></p>
</div>
<h2>1. Resumen ejecutivo</h2>
<p>{{ resumen }}</p>
<div class="kpis">{% for n, v in kpi_cards %}<div class="kpi"><div class="n">{{ n }}</div><div class="v">{{ v }}</div></div>{% endfor %}</div>
<h2>2. Metodología y fuente</h2>
<p><b>Fuente:</b> {{ fuente }}. Pipeline: parquet enriquecido → agregados anónimos (pandas) → KPIs → gráficos (Plotly) → plantilla Jinja2 → HTML/PDF/DOCX. Validaciones automáticas: las sumas por etiqueta, día y protocolo coinciden con el total de flujos.</p>
<h2>3. Resultados</h2>
{% for titulo, img in figs %}<figure><img src="{{ img }}" alt="{{ titulo }}"><figcaption>Figura {{ loop.index }}. {{ titulo }}. Fuente: {{ fuente }}.</figcaption></figure>{% endfor %}
<h2>Detalle por tipo de ataque</h2>
<table><tr><th>Tipo de ataque</th><th>Familia</th><th>Flujos</th><th>% de ataques</th><th>Duración mediana (s)</th></tr>
{% for r in tabla_atq %}<tr><td>{{ r.label }}</td><td>{{ r.familia }}</td><td>{{ "{:,}".format(r.flujos) }}</td><td>{{ "%.2f"|format(r.flujos / k.ataques * 100) }}%</td><td>{{ "%.4g"|format(r.duracion_mediana_s) }}</td></tr>{% endfor %}</table>
<h2>4. Hallazgos</h2>
{% for h in hallazgos %}<div class="hallazgo"><h3>{{ loop.index }}. {{ h.titulo }}</h3>
<p><b>Evidencia:</b> {{ h.evidencia }}</p><p><b>Interpretación:</b> {{ h.interpretacion }}</p>
<p><b>Impacto:</b> {{ h.impacto }}</p><p><b>Decisión sugerida:</b> {{ h.decision }}</p><p><b>Limitación:</b> {{ h.limitacion }}</p></div>{% endfor %}
<h2>5. Recomendaciones</h2>
<table><tr><th>#</th><th>Acción</th><th>Prioridad</th><th>Responsable</th><th>Plazo</th><th>KPI de seguimiento</th></tr>
{% for r in recomendaciones %}<tr><td>{{ loop.index }}</td>{% for c in r %}<td>{{ c }}</td>{% endfor %}</tr>{% endfor %}</table>
<h2>6. Limitaciones y gobernanza</h2>
{% for l in limitaciones %}<div class="limit">{{ l }}</div>{% endfor %}
<p class="meta">Informe generado automáticamente desde el notebook Reporte_Automatizado.ipynb. Trazabilidad en metadata_informe.json.</p>
</body></html>"""

(TPL / "informe.html.j2").write_text(PLANTILLA, encoding="utf-8")
print("Plantilla guardada en", TPL / "informe.html.j2")

Plantilla guardada en ../Reporteria/templates/informe.html.j2


GENERACIÓN DE HTML, PDF Y DOCX

In [9]:
b64 = lambda ruta: "data:image/png;base64," + base64.b64encode(Path(ruta).read_bytes()).decode()

FIG_TITULOS = [("g1_tipos_ataque", "Ataques por tipo"), ("g2_ataques_por_dia", "Ataques por día"),
            ("g3_ataques_por_hora", "Ataques por hora"), ("g4_familias", "Familias de ataque"),
            ("g5_volumen_riesgo", "Riesgo por volumen de paquetes"), ("g6_protocolo", "Flujos por protocolo"),
            ("g7_duracion_por_ataque", "Duración mediana por tipo de ataque")]

kpi_cards = [("Flujos analizados", fmt(k["total_flujos"])), ("Flujos de ataque", fmt(k["ataques"])),
            ("% tráfico malicioso", f"{k['pct_malicioso']:.2f}%"), ("Tipos de ataque detectados", str(k["tipos_ataque"])),
            (f"Peso de {k['familia_top']}", f"{k['familia_top_pct']:.1f}%"), ("Ataques de bajo volumen", f"{k['pct_atq_volumen_bajo']:.1f}%")]

resumen = (f"Se analizaron {fmt(k['total_flujos'])} flujos de red entre {k['periodo']}. El {k['pct_malicioso']:.2f}% ({fmt(k['ataques'])}) fue malicioso, "
            f"repartido en {k['tipos_ataque']} tipos de ataque. {k['top_label']} es el más frecuente ({k['top_label_pct']:.1f}% de los ataques) y la familia "
            f"{k['familia_top']} concentra {k['familia_top_pct']:.1f}%. La hora {k['hora_pico']}:00 reúne {k['hora_pico_pct']:.1f}% de los ataques, "
            f"{k['pct_atq_volumen_bajo']:.1f}% son de bajo volumen y sus flujos duran ≈{k['factor_duracion']:.0f} veces más que los benignos (mediana), "
            f"por lo que las reglas de detección deben combinar variables y no depender solo del tamaño del flujo.")

ctx = dict(k=k, kpi_cards=kpi_cards, resumen=resumen, hallazgos=hallazgos, recomendaciones=RECOMENDACIONES,
            limitaciones=LIMITACIONES, version=VERSION, equipo=EQUIPO, periodo=k["periodo"], fuente=FUENTE_TXT,
            fecha_generacion=datetime.now().strftime("%d/%m/%Y %H:%M"),
            figs=[(t, b64(RUTAS[n])) for n, t in FIG_TITULOS],
            tabla_atq=solo_atq.sort_values("flujos", ascending=False).to_dict("records"))

# HTML
env = Environment(loader=FileSystemLoader(str(TPL)), autoescape=True)
html = env.get_template("informe.html.j2").render(**ctx)
R_HTML = OUT / "informe_ciberseguridad.html"
R_HTML.write_text(html, encoding="utf-8")

# PDF
def buscar_navegador():
    for var in ("CHROME_PATH", "BROWSER_PATH"):
        if os.environ.get(var) and Path(os.environ[var]).exists():
            return os.environ[var]
    for nombre in ("chrome", "google-chrome", "chromium", "msedge", "microsoft-edge"):
        p = shutil.which(nombre)
        if p: return p
    for p in (r"C:\Program Files\Google\Chrome\Application\chrome.exe", r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
            r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe", r"C:\Program Files\Microsoft\Edge\Application\msedge.exe"):
        if Path(p).exists(): return p
    return None

def generar_pdf(html_path, pdf_path):
    if not os.environ.get("FORZAR_NAVEGADOR"):
        try:
            from weasyprint import HTML
            HTML(filename=str(html_path)).write_pdf(str(pdf_path))
            return "WeasyPrint"
        except Exception as e:
            print("WeasyPrint no disponible, se usa el navegador:", type(e).__name__)
    nav = buscar_navegador()
    if not nav:
        raise RuntimeError("Instala WeasyPrint o Chrome/Edge para generar el PDF (o abre el HTML y usa Imprimir > Guardar como PDF).")
    cmd = [nav, "--headless=new", "--no-sandbox", "--disable-gpu", "--no-pdf-header-footer",
        f"--print-to-pdf={Path(pdf_path).resolve()}", Path(html_path).resolve().as_uri()]
    subprocess.run(cmd, check=True, capture_output=True, timeout=180)
    return "Chrome/Edge headless"

R_PDF = OUT / f"informe_ciberseguridad_v{VERSION}.pdf"
motor = generar_pdf(R_HTML, R_PDF)

# DOCX
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()
doc.add_heading("INFORME EJECUTIVO DE SEGURIDAD DE RED", 0).alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_paragraph(f"Periodo analizado: {k['periodo']}\nFecha de generación: {ctx['fecha_generacion']}\nVersión: {VERSION}\nEquipo: {EQUIPO}\nFuente: {FUENTE_TXT}")
doc.add_heading("1. Resumen ejecutivo", 1); doc.add_paragraph(resumen)
doc.add_heading("2. Indicadores clave", 1)
tb = doc.add_table(rows=1, cols=2); tb.style = "Light Grid Accent 1"
tb.rows[0].cells[0].text, tb.rows[0].cells[1].text = "Indicador", "Valor"
for n, v in kpi_cards:
    c = tb.add_row().cells; c[0].text, c[1].text = n, v
doc.add_heading("3. Resultados", 1)
for i, (n, t) in enumerate(FIG_TITULOS, 1):
    doc.add_heading(f"3.{i} {t}", 2); doc.add_picture(str(RUTAS[n]), width=Inches(6))
doc.add_heading("4. Hallazgos", 1)
for i, h in enumerate(hallazgos, 1):
    doc.add_heading(f"{i}. {h['titulo']}", 2)
    for lbl, key in [("Evidencia", "evidencia"), ("Interpretación", "interpretacion"), ("Impacto", "impacto"), ("Decisión", "decision"), ("Limitación", "limitacion")]:
        p = doc.add_paragraph(); p.add_run(f"{lbl}: ").bold = True; p.add_run(h[key])
doc.add_heading("5. Recomendaciones", 1)
tr = doc.add_table(rows=1, cols=6); tr.style = "Light Grid Accent 1"
for i, hd in enumerate(["#", "Acción", "Prioridad", "Responsable", "Plazo", "KPI de seguimiento"]):
    tr.rows[0].cells[i].text = hd
for i, rec in enumerate(RECOMENDACIONES, 1):
    c = tr.add_row().cells; c[0].text = str(i)
    for j, val in enumerate(rec, 1): c[j].text = val
doc.add_heading("6. Limitaciones", 1)
for l in LIMITACIONES: doc.add_paragraph(l, style="List Bullet")
R_DOCX = OUT / f"informe_ciberseguridad_v{VERSION}.docx"
doc.save(str(R_DOCX))

for p in (R_HTML, R_PDF, R_DOCX):
    print(f"OK  {p.name:<42}{p.stat().st_size/1024:>9,.1f} KB")
print("PDF generado con:", motor)

OK  informe_ciberseguridad.html                 1,255.7 KB
OK  informe_ciberseguridad_v1.0.pdf               657.9 KB
OK  informe_ciberseguridad_v1.0.docx              794.3 KB
PDF generado con: WeasyPrint


METADATA DE TRAZABILIDAD Y VERIFICACIÓN DE LA GOBERNANZA

In [10]:
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

# Verificación de gobernanza: ninguna IP en agregados, HTML, DOCX ni JSON 
import zipfile
PATRON_IP = re.compile(r"(?<![\d.])(?:\d{1,3}\.){3}\d{1,3}(?![\d.])")
html_sin_img = re.sub(r"data:image/png;base64,[A-Za-z0-9+/=]+", "", R_HTML.read_text(encoding="utf-8"))
docx_xml = zipfile.ZipFile(R_DOCX).read("word/document.xml").decode("utf-8")
hallazgos_ip = {"html": PATRON_IP.findall(html_sin_img), "docx": PATRON_IP.findall(docx_xml)}
for csv in AGG.glob("*.csv"):
    hallazgos_ip[csv.name] = PATRON_IP.findall(csv.read_text(encoding="utf-8"))
    assert not {"flow_id", "source_ip", "destination_ip", "source_port"} & set(pd.read_csv(csv, nrows=0).columns), f"Columna prohibida en {csv.name}"
con_ip = {n: v for n, v in hallazgos_ip.items() if v}
assert not con_ip, f"Se detectaron IP en: {con_ip}"
print("Gobernanza OK: sin IP, flow_id ni puertos de origen en ninguna salida.")

# metadata_informe.json
meta = {
    "version": VERSION, "fecha_generacion": datetime.now().isoformat(timespec="seconds"), "modo_ejecucion": MODO,
    "periodo_analizado": k["periodo"],
    "fuentes_datos": [FUENTE_TXT] + [f"datos/agregados/{p.name}" for p in sorted(AGG.glob("*.csv"))],
    "filtros_aplicados": {"periodo": "completo", "etiquetas": "todas", "ips_publicadas": False},
    "kpis": {key: (round(val, 4) if isinstance(val, float) else val) for key, val in k.items()},
    "validaciones": ["suma por etiqueta == total", "suma por día == total", "ataques por día == ataques",
                    "suma por protocolo == total", "sin IP/flow_id/puerto origen en las salidas"],
    "archivos_generados": {p.name: {"sha256_16": sha(p), "bytes": p.stat().st_size} for p in (R_HTML, R_PDF, R_DOCX)},
    "graficos": [p.name for p in RUTAS.values()], "motor_pdf": motor,
    "generado_por": "Reporte_Automatizado.ipynb", "gobernanza": "Sin IP, flow_id ni puertos de origen (Guía 4)",
}
(OUT / "metadata_informe.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("metadata_informe.json guardado")
print(json.dumps({k_: meta[k_] for k_ in ("version", "modo_ejecucion", "periodo_analizado", "archivos_generados")}, indent=2, ensure_ascii=False))

Gobernanza OK: sin IP, flow_id ni puertos de origen en ninguna salida.
metadata_informe.json guardado
{
  "version": "1.0",
  "modo_ejecucion": "parquet",
  "periodo_analizado": "03/07/2017 – 07/07/2017",
  "archivos_generados": {
    "informe_ciberseguridad.html": {
      "sha256_16": "b41e0b80db1f7b7c",
      "bytes": 1285818
    },
    "informe_ciberseguridad_v1.0.pdf": {
      "sha256_16": "5d6196e341b3033e",
      "bytes": 673694
    },
    "informe_ciberseguridad_v1.0.docx": {
      "sha256_16": "1f7c8b5139329cfa",
      "bytes": 813353
    }
  }
}


AUTOMATIZACIÓN CON GITHUB ACTIONS

In [11]:
WORKFLOW = """name: Generar informe automático

on:
    schedule:
    - cron: '0 8 * * 1'      # todos los lunes 08:00 UTC
    workflow_dispatch:
        push:
    paths: ['datos/agregados/**', 'Reporte_Automatizado.ipynb', 'templates/**']

jobs:
generar-informe:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
        with:
        python-version: '3.11'
        cache: pip
    - name: Dependencias del sistema (WeasyPrint)
        run: sudo apt-get update && sudo apt-get install -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz-subset0
    - name: Dependencias Python
        run: pip install -r requirements.txt jupyter nbconvert ipykernel
    - name: Chrome para exportar gráficos Plotly
        run: plotly_get_chrome -y
    - name: Ejecutar notebook (modo agregados)
        env:
        REPORTE_VERSION: "1.${{ github.run_number }}"
        run: jupyter nbconvert --to notebook --execute Reporte_Automatizado.ipynb --output ejecutado.ipynb
    - name: Subir artefactos
        uses: actions/upload-artifact@v4
        with:
        name: informe-generado
        path: |
            exportaciones/
            evidencias/reporte/
"""
REQ = "pandas>=2.0\npyarrow>=14\nplotly>=5.20\nkaleido>=1.0\njinja2>=3.1\nweasyprint>=62\npython-docx>=1.1\n"
README = """# Reportería automatizada — Cybersecurity Data Analytics
Notebook `Reporte_Automatizado.ipynb`: parquet enriquecido -> agregados anónimos -> KPIs -> Plotly -> Jinja2 -> HTML/PDF/DOCX + metadata JSON.
- `datos/agregados/`: CSV anónimos (se suben a GitHub). El parquet completo NO se sube.
- `templates/informe.html.j2`: plantilla. `exportaciones/`: informes. `evidencias/reporte/`: gráficos.
- Local: abrir el notebook y ejecutar todo (busca el parquet vía Configuracion.py). GitHub: Actions > Run workflow.
Gobernanza: sin IP, flow_id ni puertos de origen. Fuente: CICIDS2017 - https://www.unb.ca/cic/datasets/ids-2017.html
"""
BITACORA = """# Bitácora de validación del informe
| Verificación | Resultado |
|---|---|
| Suma por etiqueta / día / protocolo == total de flujos | OK (assert automático en el notebook) |
| Ataques por día == total de ataques | OK |
| Sin IP, flow_id ni puerto de origen en CSV, HTML, DOCX | OK (verificación automática) |
| KPIs coinciden con Power BI (19.68% malicioso, 556,555 ataques, DoS/DDoS 68.23%) | Revisar contra el dashboard |
"""
(BASE / ".github/workflows/generar_informe.yml").write_text(WORKFLOW, encoding="utf-8")
(BASE / "requirements.txt").write_text(REQ, encoding="utf-8")
(BASE / "README.md").write_text(README, encoding="utf-8")
(BASE / "documentos/bitacora_validacion.md").write_text(BITACORA, encoding="utf-8")
print("Creados: workflow, requirements.txt, README.md y bitácora en", BASE.resolve())

Creados: workflow, requirements.txt, README.md y bitácora en /home/vctor/Proyecto_GDIABD_Cybersecurity_Data_Analytics/Reporteria
